In [1]:
import os
import gc
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd
from functools import partial
import torch.nn.functional as F


# import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler

from utils import *
os.makedirs('./phi4_8x8', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
def conv3x3(in_planes, out_planes, stride=1, bias=False):
    "3x3 convolution with padding"
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=bias)


def conv1x1(in_planes, out_planes, stride=1, bias=False):
    "1x1 convolution"
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride,
                     padding=0, bias=bias)


def dilated_conv3x3(in_planes, out_planes, dilation, bias=True):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, padding=dilation, dilation=dilation, bias=bias)


class ConditionalBatchNorm2d(nn.Module):
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features = num_features
        self.bias = bias
        self.bn = nn.BatchNorm2d(num_features, affine=False)
        if self.bias:
            self.embed = nn.Embedding(num_classes, num_features * 2)
            self.embed.weight.data[:, :num_features].uniform_()  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, num_features:].zero_()  # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, num_features)
            self.embed.weight.data.uniform_()

    def forward(self, x, y):
        out = self.bn(x)
        if self.bias:
            gamma, beta = self.embed(y).chunk(2, dim=1)
            out = gamma.view(-1, self.num_features, 1, 1) * out + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma = self.embed(y)
            out = gamma.view(-1, self.num_features, 1, 1) * out
        return out


class ConditionalInstanceNorm2d(nn.Module):
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features = num_features
        self.bias = bias
        self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
        if bias:
            self.embed = nn.Embedding(num_classes, num_features * 2)
            self.embed.weight.data[:, :num_features].uniform_()  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, num_features:].zero_()  # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, num_features)
            self.embed.weight.data.uniform_()

    def forward(self, x, y):
        h = self.instance_norm(x)
        if self.bias:
            gamma, beta = self.embed(y).chunk(2, dim=-1)
            out = gamma.view(-1, self.num_features, 1, 1) * h + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma = self.embed(y)
            out = gamma.view(-1, self.num_features, 1, 1) * h
        return out


class CRPBlock(nn.Module):
    def __init__(self, features, n_stages, act=nn.ReLU()):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(n_stages):
            self.convs.append(conv3x3(features, features, stride=1, bias=False))
        self.n_stages = n_stages
        self.maxpool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)
        self.act = act

    def forward(self, x):
        x = self.act(x)
        path = x
        for i in range(self.n_stages):
            path = self.maxpool(path)
            path = self.convs[i](path)
            x = path + x
        return x


class CondCRPBlock(nn.Module):
    def __init__(self, features, n_stages, num_classes, normalizer, act=nn.ReLU()):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(n_stages):
            self.norms.append(normalizer(features, num_classes, bias=True))
            self.convs.append(conv3x3(features, features, stride=1, bias=False))
        self.n_stages = n_stages
        self.maxpool = nn.AvgPool2d(kernel_size=5, stride=1, padding=2)
        self.act = act

    def forward(self, x, y):
        x = self.act(x)
        path = x
        for i in range(self.n_stages):
            path = self.norms[i](path, y)
            path = self.maxpool(path)
            path = self.convs[i](path)
            x = path + x
        return x


class CondRCUBlock(nn.Module):
    def __init__(self, features, n_blocks, n_stages, num_classes, normalizer, act=nn.ReLU()):
        super().__init__()

        for i in range(n_blocks):
            for j in range(n_stages):
                setattr(self, '{}_{}_norm'.format(i + 1, j + 1), normalizer(features, num_classes, bias=True))
                setattr(self, '{}_{}_conv'.format(i + 1, j + 1),
                        conv3x3(features, features, stride=1, bias=False))

        self.stride = 1
        self.n_blocks = n_blocks
        self.n_stages = n_stages
        self.act = act

    def forward(self, x, y):
        for i in range(self.n_blocks):
            residual = x
            for j in range(self.n_stages):
                x = getattr(self, '{}_{}_norm'.format(i + 1, j + 1))(x, y)
                x = self.act(x)
                x = getattr(self, '{}_{}_conv'.format(i + 1, j + 1))(x)
            x += residual
        return x


class CondMSFBlock(nn.Module):
    def __init__(self, in_planes, features, num_classes, normalizer):
        """
        :param in_planes: tuples of input planes
        """
        super().__init__()
        assert isinstance(in_planes, list) or isinstance(in_planes, tuple)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.features = features

        for i in range(len(in_planes)):
            self.convs.append(conv3x3(in_planes[i], features, stride=1, bias=True))
            self.norms.append(normalizer(in_planes[i], num_classes, bias=True))

    def forward(self, xs, y, shape):
        sums = torch.zeros(xs[0].shape[0], self.features, *shape, device=xs[0].device)
        for i in range(len(self.convs)):
            h = self.norms[i](xs[i], y)
            h = self.convs[i](h)
            h = F.interpolate(h, size=shape, mode='bilinear', align_corners=True)
            sums += h
        return sums


class CondRefineBlock(nn.Module):
    def __init__(self, in_planes, features, num_classes, normalizer, act=nn.ReLU(), start=False, end=False):
        super().__init__()

        assert isinstance(in_planes, tuple) or isinstance(in_planes, list)
        self.n_blocks = n_blocks = len(in_planes)

        self.adapt_convs = nn.ModuleList()
        for i in range(n_blocks):
            self.adapt_convs.append(
                CondRCUBlock(in_planes[i], 2, 2, num_classes, normalizer, act)
            )

        self.output_convs = CondRCUBlock(features, 3 if end else 1, 2, num_classes, normalizer, act)

        if not start:
            self.msf = CondMSFBlock(in_planes, features, num_classes, normalizer)

        self.crp = CondCRPBlock(features, 2, num_classes, normalizer, act)

    def forward(self, xs, y, output_shape):
        assert isinstance(xs, tuple) or isinstance(xs, list)
        hs = []
        for i in range(len(xs)):
            h = self.adapt_convs[i](xs[i], y)
            hs.append(h)

        if self.n_blocks > 1:
            h = self.msf(hs, y, output_shape)
        else:
            h = hs[0]

        h = self.crp(h, y)
        h = self.output_convs(h, y)

        return h


class ConvMeanPool(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True, adjust_padding=False):
        super().__init__()
        if not adjust_padding:
            self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
        else:
            self.conv = nn.Sequential(
                nn.ZeroPad2d((1, 0, 1, 0)),
                nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
            )

    def forward(self, inputs):
        output = self.conv(inputs)
        output = sum(
            [output[:, :, ::2, ::2], output[:, :, 1::2, ::2], output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
        return output


class MeanPoolConv(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
        super().__init__()
        self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)

    def forward(self, inputs):
        output = inputs
        output = sum(
            [output[:, :, ::2, ::2], output[:, :, 1::2, ::2], output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
        return self.conv(output)


class UpsampleConv(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
        super().__init__()
        self.conv = nn.Conv2d(input_dim, output_dim, kernel_size, stride=1, padding=kernel_size // 2, bias=biases)
        self.pixelshuffle = nn.PixelShuffle(upscale_factor=2)

    def forward(self, inputs):
        output = inputs
        output = torch.cat([output, output, output, output], dim=1)
        output = self.pixelshuffle(output)
        return self.conv(output)


class ConditionalResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, num_classes, resample=None, act=nn.ELU(),
                 normalization=ConditionalBatchNorm2d, adjust_padding=False, dilation=None):
        super().__init__()
        self.non_linearity = act
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.resample = resample
        self.pad = nn.CircularPad2d(1)

        if resample == 'down':
            if dilation is not None:
                self.conv1 = dilated_conv3x3(input_dim, input_dim, dilation=dilation)
                self.normalize2 = normalization(input_dim, num_classes)
                self.conv2 = dilated_conv3x3(input_dim, output_dim, dilation=dilation)
                conv_shortcut = partial(dilated_conv3x3, dilation=dilation)
            else:
                self.conv1 = nn.Conv2d(input_dim, input_dim, 3, stride=1, padding=1)
                self.normalize2 = normalization(input_dim, num_classes)
                self.conv2 = ConvMeanPool(input_dim, output_dim, 3, adjust_padding=adjust_padding)
                conv_shortcut = partial(ConvMeanPool, kernel_size=1, adjust_padding=adjust_padding)

        elif resample is None:
            if dilation is not None:
                conv_shortcut = partial(dilated_conv3x3, dilation=dilation)
                self.conv1 = dilated_conv3x3(input_dim, output_dim, dilation=dilation)
                self.normalize2 = normalization(output_dim, num_classes)
                self.conv2 = dilated_conv3x3(output_dim, output_dim, dilation=dilation)
            else:
                conv_shortcut = nn.Conv2d
                self.conv1 = nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=1, padding=1)
                self.normalize2 = normalization(output_dim, num_classes)
                self.conv2 = nn.Conv2d(output_dim, output_dim, kernel_size=3, stride=1, padding=1)
        else:
            raise Exception('invalid resample value')

        if output_dim != input_dim or resample is not None:
            self.shortcut = conv_shortcut(input_dim, output_dim)

        self.normalize1 = normalization(input_dim, num_classes)

    def forward(self, x, y):
        output = self.normalize1(x, y)
        output = self.non_linearity(output)
        output = self.conv1(output)
        output = self.normalize2(output, y)
        output = self.non_linearity(output)
        output = self.conv2(output)

        if self.output_dim == self.input_dim and self.resample is None:
            shortcut = x
        else:
            shortcut = self.shortcut(x)

        return shortcut + output


class ConditionalInstanceNorm2dPlus(nn.Module):
    def __init__(self, num_features, num_classes, bias=True):
        super().__init__()
        self.num_features = num_features
        self.bias = bias
        self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
        if bias:
            self.embed = nn.Embedding(num_classes, num_features * 3)
            self.embed.weight.data[:, :2 * num_features].normal_(1, 0.02)  # Initialise scale at N(1, 0.02)
            self.embed.weight.data[:, 2 * num_features:].zero_()  # Initialise bias at 0
        else:
            self.embed = nn.Embedding(num_classes, 2 * num_features)
            self.embed.weight.data.normal_(1, 0.02)

    def forward(self, x, y):
        means = torch.mean(x, dim=(2, 3))
        m = torch.mean(means, dim=-1, keepdim=True)
        v = torch.var(means, dim=-1, keepdim=True)
        means = (means - m) / (torch.sqrt(v + 1e-5))
        h = self.instance_norm(x)

        if self.bias:
            gamma, alpha, beta = self.embed(y).chunk(3, dim=-1)
            h = h + means[..., None, None] * alpha[..., None, None]
            out = gamma.view(-1, self.num_features, 1, 1) * h + beta.view(-1, self.num_features, 1, 1)
        else:
            gamma, alpha = self.embed(y).chunk(2, dim=-1)
            h = h + means[..., None, None] * alpha[..., None, None]
            out = gamma.view(-1, self.num_features, 1, 1) * h
        return out


class CondRefineNetDilated(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.logit_transform = config.data.logit_transform
        # self.norm = ConditionalInstanceNorm2d
        self.norm = ConditionalInstanceNorm2dPlus
        self.ngf = ngf = config.model.ngf
        self.num_classes = config.model.num_classes
        self.act = act = nn.ELU()
        # self.act = act = nn.ReLU(True)
        
        # self.pad = nn.CircularPad2d(1)
        self.begin_conv = nn.Conv2d(config.data.channels, ngf, 3, stride=1, padding=1)
        self.normalizer = self.norm(ngf, self.num_classes)

        self.end_conv = nn.Conv2d(ngf, config.data.channels, 3, stride=1, padding=1)

        self.res1 = nn.ModuleList([
            ConditionalResidualBlock(self.ngf, self.ngf, self.num_classes, resample=None, act=act,
                                     normalization=self.norm),
            ConditionalResidualBlock(self.ngf, self.ngf, self.num_classes, resample=None, act=act,
                                     normalization=self.norm)]
        )

        self.res2 = nn.ModuleList([
            ConditionalResidualBlock(self.ngf, 2 * self.ngf, self.num_classes, resample='down', act=act,
                                     normalization=self.norm),
            ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample=None, act=act,
                                     normalization=self.norm)]
        )

        self.res3 = nn.ModuleList([
            ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample='down', act=act,
                                     normalization=self.norm, dilation=2),
            ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample=None, act=act,
                                     normalization=self.norm, dilation=2)]
        )

        if config.data.image_size == 28:
            self.res4 = nn.ModuleList([
                ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample='down', act=act,
                                         normalization=self.norm, adjust_padding=True, dilation=4),
                ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample=None, act=act,
                                         normalization=self.norm, dilation=4)]
            )
        else:
            self.res4 = nn.ModuleList([
                ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample='down', act=act,
                                         normalization=self.norm, adjust_padding=False, dilation=4),
                ConditionalResidualBlock(2 * self.ngf, 2 * self.ngf, self.num_classes, resample=None, act=act,
                                         normalization=self.norm, dilation=4)]
            )

        self.refine1 = CondRefineBlock([2 * self.ngf], 2 * self.ngf, self.num_classes, self.norm, act=act, start=True)
        self.refine2 = CondRefineBlock([2 * self.ngf, 2 * self.ngf], 2 * self.ngf, self.num_classes, self.norm, act=act)
        self.refine3 = CondRefineBlock([2 * self.ngf, 2 * self.ngf], self.ngf, self.num_classes, self.norm, act=act)
        self.refine4 = CondRefineBlock([self.ngf, self.ngf], self.ngf, self.num_classes, self.norm, act=act, end=True)

    def _compute_cond_module(self, module, x, y):
        for m in module:
            x = m(x, y)
        return x

    def forward(self, x, y):
        if not self.logit_transform:
            x = 2 * x - 1.

        output = self.begin_conv(x)

        layer1 = self._compute_cond_module(self.res1, output, y)
        layer2 = self._compute_cond_module(self.res2, layer1, y)
        layer3 = self._compute_cond_module(self.res3, layer2, y)
        layer4 = self._compute_cond_module(self.res4, layer3, y)

        ref1 = self.refine1([layer4], y, layer4.shape[2:])
        ref2 = self.refine2([layer3, ref1], y, layer3.shape[2:])
        ref3 = self.refine3([layer2, ref2], y, layer2.shape[2:])
        output = self.refine4([layer1, ref3], y, layer1.shape[2:])

        output = self.normalizer(output, y)
        output = self.act(output)
        output = self.end_conv(output)
        return output


In [5]:
from types import SimpleNamespace

config = SimpleNamespace()
config.data = SimpleNamespace()
config.data.image_size = 8
config.data.logit_transform = False

config.model = SimpleNamespace()
config.model.ngf = 64
config.model.num_classes = 5
config.data.channels = 1

In [6]:
# class Score(nn.Module):
#     def __init__(self, input_shape, num_classes, hidden_dim):
#         super().__init__()
#         self.input_shape = input_shape
#         self.hidden_dim = hidden_dim
#         self.num_classes = num_classes
        
#         self.conv1 = nn.Conv2d(self.input_shape[0], self.hidden_dim, kernel_size=3, stride=1)
#         self.norm1 = ConditionalBatchNorm2d(self.hidden_dim, self.num_classes)
#         self.conv2 = nn.Conv2d(self.hidden_dim, self.hidden_dim, kernel_size=3, stride=1)
#         self.norm2 = ConditionalBatchNorm2d(self.hidden_dim, self.num_classes)
#         self.conv3 = nn.Conv2d(self.hidden_dim, self.hidden_dim * 2, kernel_size=3, stride=1)
#         self.norm3 = ConditionalBatchNorm2d(self.hidden_dim * 2, self.num_classes)
#         self.conv4 = nn.Conv2d(self.hidden_dim * 2, self.hidden_dim * 2, kernel_size=3, stride=1)
#         self.norm4 = ConditionalBatchNorm2d(self.hidden_dim * 2, self.num_classes)
#         self.conv5 = nn.Conv2d(self.hidden_dim * 2, self.hidden_dim, kernel_size=3, stride=1)
#         self.norm5 = ConditionalBatchNorm2d(self.hidden_dim, self.num_classes)
#         self.conv6 = nn.Conv2d(self.hidden_dim, self.input_shape[0], kernel_size=3, stride=1)

#         self.pad = nn.CircularPad2d(1)
#         self.tanh = nn.Tanh()

#     def forward(self, x, y):
#         x = self.conv1(x)
#         x = self.norm1(x, y)
#         x = self.tanh(x)
#         x = self.pad(x)

#         x = self.conv2(x)
#         x = self.norm2(x, y)
#         x = self.tanh(x)
#         x = self.pad(x)

#         x = self.conv3(x)
#         x = self.norm3(x, y)
#         x = self.tanh(x)
#         x = self.pad(x)

#         x = self.conv4(x)
#         x = self.norm4(x, y)
#         x = self.tanh(x)
#         x = self.pad(x)

#         x = self.conv5(x)
#         x = self.norm5(x, y)
#         x = self.tanh(x)
#         x = self.pad(x)

#         x = self.conv6(x)
#         x = self.pad(x)

#         return x


In [7]:
def log_prob(x,m2 = 1.0 ,lamda = 4.0):
    x.requires_grad_(True)
    potential = m2*x**2 + lamda*x**4
    Nd = len(x.shape)-1
    dims = range(2,Nd+1)
    for mu in dims:
        potential += 2*x**2
        potential -= x*torch.roll(x,-2,mu)
        potential -= x*torch.roll(x,2,mu)
    return -torch.sum(potential , dim = [1,2,3]).to(device = x.device)

In [8]:
def anneal_dsm_score_estimation(scorenet, samples, labels, sigmas, anneal_power=2.):
    used_sigmas = sigmas[labels].view(samples.shape[0], *([1] * len(samples.shape[1:])))
    perturbed_samples = samples + torch.randn_like(samples) * used_sigmas
    target = - 1 / (used_sigmas ** 2) * (perturbed_samples - samples)
    # perturbed_samples.requires_grad = True
    scores = scorenet(perturbed_samples,labels)
    target = target.view(target.shape[0], -1)
    scores = scores.view(scores.shape[0], -1)
    loss = 1 / 2. * ((scores - target) ** 2).sum(dim=-1) * used_sigmas.squeeze() ** anneal_power

    return loss.mean(dim=0)

In [9]:
def mse_loss_score(scorenet,labels, sigmas,jacobian1,log_prob,samples,m2,lamda):
    used_sigmas = sigmas[labels].view(samples.shape[0], *([1] * len(samples.shape[1:])))
    perturbed_samples = samples + torch.randn_like(samples) * used_sigmas
    model_score = scorenet(perturbed_samples,labels)
    exact_score = jacobian1(log_prob,perturbed_samples,m2,lamda)
    model_score = model_score.view(model_score.shape[0], -1)
    exact_score = exact_score.view(exact_score.shape[0], -1)
    loss = 1 / 2. * ((model_score - exact_score) ** 2).sum(dim=-1).mean(dim=0)
    return loss

In [10]:
def jacobian1(f, x,m2,lamda):
    """
    Computes the Jacobian
    """
    
    y = f(x,m2,lamda)
    v = torch.zeros_like(y,device = torch_device)
    v[:] = 1.
    dy_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
    jacobian = dy_dx.requires_grad_().to(torch_device)
    return jacobian

In [11]:
dim = 64
data_filename = "phi4_8x8_lattices_for_lamda_4_mass_1_larger_interaction.pkl"
f = open(data_filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
dataset = torch.tensor(dataset , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))

In [12]:
dataset.shape

torch.Size([200000, 1, 8, 8])

In [13]:
# val_filename = "phi4_test_lattices.pkl"
# f_ = open(val_filename, 'rb')
# valset = pickle.load(f_,encoding="latin1")
valset = torch.tensor(dataset[100000:125000] , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))

/tmp/ipykernel_3371028/3135883496.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valset = torch.tensor(dataset[100000:125000] , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))


In [14]:
valset.shape

torch.Size([25000, 1, 8, 8])

In [15]:
# test_filename = "phi4_8x8_data_trained_with_100000_samples.pkl"
# k_ = open(test_filename, 'rb')
# testset = pickle.load(k_,encoding="latin1")
testset = torch.tensor(dataset[125000:150000] , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))

/tmp/ipykernel_3371028/710889222.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  testset = torch.tensor(dataset[125000:150000] , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))


In [16]:
testset.shape

torch.Size([25000, 1, 8, 8])

In [17]:
model = CondRefineNetDilated(config).to(torch_device)

In [18]:
sigmas = torch.tensor(np.exp(np.linspace(np.log(1.0), np.log(0.01),5))).float().to(torch_device)

In [51]:
batch_size = 250
no_training_samples = 100000
no_validation_samples = 25000
no_test_samples = 25000

train_dataloader = torch.utils.data.DataLoader(dataset[:100000], batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(valset, batch_size = batch_size, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(testset, batch_size = batch_size, shuffle=False)

In [38]:
PATH = "./phi4_8x8/score_estimation_model_for_lamda_4_mass_1_larger_interaction.pt"
optimizer = optim.Adam(model.parameters(),lr= 1.e-6)


In [50]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [48]:
epochs =  5

for i in range(epochs):
    model.train()
    train_score_loss = 0.
    val_score_loss = 0.
    val_mse_loss = 0.
    
    for x in train_dataloader:
        labels = torch.randint(0, len(sigmas), (x.shape[0],), device=x.device)
        loss   = anneal_dsm_score_estimation(model, x, labels, sigmas, anneal_power = 1.0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_score_loss += loss.item()*(x.shape[0])
        
  
    # model.eval()
    # with torch.no_grad():
    for x in val_dataloader:
        # test_labels = torch.randint(0, len(sigmas), (x.shape[0],), device=x.device)
        test_labels = 4*torch.ones((x.shape[0],), device=x.device,dtype = torch.int64)

        val_loss = anneal_dsm_score_estimation(model, x, test_labels, sigmas, anneal_power = 1.0)
        mse_loss = mse_loss_score(model,test_labels, sigmas,jacobian1,log_prob,x,1.0,4.0)
        val_score_loss += val_loss.item()*(x.shape[0])
        val_mse_loss += mse_loss.item()*(x.shape[0]) 
        
    avg_train_loss = train_score_loss/no_training_samples
    avg_val_loss   = val_score_loss/no_validation_samples
    avg_mse_loss   = val_mse_loss/no_validation_samples        
    template = 'Epoch {:3d}, Train loss: {:.4f},Val loss: {:.4f},Val MSE loss: {:.4f}'
    print (template.format(i+1 , avg_train_loss , avg_val_loss , avg_mse_loss))
    

Epoch   1, Train loss: 902.2006,Val loss: 3197.3409,Val MSE loss: 5.6755
Epoch   2, Train loss: 903.2645,Val loss: 3194.6228,Val MSE loss: 5.5188
Epoch   3, Train loss: 907.1879,Val loss: 3201.0050,Val MSE loss: 5.8610
Epoch   4, Train loss: 904.3285,Val loss: 3194.5777,Val MSE loss: 5.5179
Epoch   5, Train loss: 906.1697,Val loss: 3191.1099,Val MSE loss: 5.8131


In [52]:
no_test_samples  = 25000
test_score_loss = 0.

for x in test_dataloader:
    test_labels = 4*torch.ones((x.shape[0],), device=x.device,dtype = torch.int64)
    test_loss = mse_loss_score(model,test_labels, sigmas,jacobian1,log_prob,x,1.0,4.0)
    test_score_loss += test_loss.item()*(x.shape[0]) 
avg_test_loss   = test_score_loss/no_test_samples
print(f"Test Score Loss:{avg_test_loss:.4f}")

Test Score Loss:5.4005


In [47]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [53]:
neg_score = []
samples = []
for x in train_dataloader:
    labels = 4*torch.ones((x.shape[0],), device=x.device,dtype = torch.int64) 
    used_sigmas = sigmas[labels].view(x.shape[0], *([1] * len(x.shape[1:])))
    perturbed_samples = x + torch.randn_like(x) * used_sigmas
    score = model(perturbed_samples,labels).cpu().detach()
    neg_score.append(-score)
    samples.append(perturbed_samples.cpu().detach())
    

In [54]:
neg_score = np.array(neg_score).reshape((-1,8,8,1))
samples   = np.array(samples).reshape((-1,8,8,1))

In [55]:
array = [samples,neg_score]

In [56]:
output_new = open('phi4_8x8_dsm_score_for_lamda_4_mass_1_larger_interaction.pkl', 'wb')
pickle.dump(array, output_new)
output_new.close()

In [57]:
samples.shape

(100000, 8, 8, 1)

In [58]:
neg_score.shape

(100000, 8, 8, 1)